# 1. Define constants

In [12]:
VIBLO_ARTICLE_URL = 'https://viblo.asia/newest?page={page}' # Default page = 1

# leave page empty
url = VIBLO_ARTICLE_URL.format(page='')

# or fill it later
url = VIBLO_ARTICLE_URL.format(page=2)

In [34]:
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag
from typing import Optional, List, Dict
import urllib.parse
import json
import datetime

def get_avaiable_articles(page: int = 1, verbose: bool = True, auto_save: str = '') -> List[Dict]:
    # Fetch articles from the API
    if verbose:
        print(f'Fetching articles from page {page}...')
    api_url = VIBLO_ARTICLE_URL.format(page=page)

    # Get the API response
    response = requests.get(api_url)

    if response.status_code != 200:
        raise Exception(f"Failed to fetch articles: {response.status_code}")

    # Parse the HTML content
    soup = BeautifulSoup(response.text, 'html.parser')

    # Get all article wrapper elements, contains link to the article and tags
    """
    Example HTML structure:
    <div class="post-title--inline">
        <h3 class="word-break mr-05">
            <a href="/p/homelab-19-cai-dat-apache-guacamole-pPLkNN3ZJRZ" class="link">[Homelab] #19 Cài đặt Apache Guacamole</a>
        </h3>
        <div class="tags d-flex flex-wrap" data-v-4365a2a0>
            <a href="/tags/homelab" class="el-tag tag el-tag--info el-tag--mini" data-v-22b6e812 data-v-4365a2a0>homelab</a>
        </div>
    </div>
    """
    ARTICLE_WRAPPER_CLASS = 'post-title--inline'
    ARTICLE_LINK_CLASS    = 'link'
    ARTICLE_TAG_CLASS     = 'el-tag tag el-tag--info el-tag--mini'

    article_elements = soup.find_all(class_=ARTICLE_WRAPPER_CLASS)

    if verbose:
        print(f'Found {len(article_elements)} articles on page {page}.')

    crawled_articles = []

    for article in article_elements:
        # Get class=link elements inside article_elements
        link_element: Optional[Tag] = article.find(class_=ARTICLE_LINK_CLASS)

        # Get all tags into a list from class='el-tag tag el-tag--info el-tag--mini'
        tags = []
        tag_elements = article.find_all(class_=ARTICLE_TAG_CLASS)
        for tag_element in tag_elements:
            tags.append(tag_element.get_text(strip=True))

        # If there is no link element, skip this article to avoid None attribute/subscript errors
        if not link_element:
            # Optionally log or collect placeholders instead of skipping
            continue

        # Use safe accessors: get_text and .get('href') to avoid 'None' / subscript issues
        name = link_element.get_text(strip=True)

        # BeautifulSoup attribute access can return different types (e.g., list-like).
        # Normalize href to a plain string before passing to urljoin to satisfy type checkers.
        href_attr = link_element.get('href')
        if isinstance(href_attr, list):
            href = href_attr[0] if href_attr else ''
        else:
            href = href_attr or ''

        full_link = urllib.parse.urljoin('https://viblo.asia', str(href))

        crawled_articles.append({
            'name': name,
            'link': full_link,
            'tags': tags
        })

    if auto_save != '':
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        with open(f'{auto_save}/crawled_articles_page_{page}_{timestamp}.json', 'w', encoding='utf-8') as f:
            json.dump(crawled_articles, f, ensure_ascii=False, indent=4)
        print(f'Auto-saved crawled articles to {auto_save}/crawled_articles_page_{page}_{timestamp}.json')

    return crawled_articles


get_avaiable_articles(auto_save='../data')

Fetching articles from page 1...
Found 21 articles on page 1.
Auto-saved crawled articles to ../data/crawled_articles_page_1_20251024_145347.json


[{'name': 'VIBLO MOBILE APP CHÍNH THỨC RA MẮT – TRẢI NGHIỆM NGAY VÀ THAM GIA MINIGAME HẤP DẪN! 📲',
  'link': 'https://viblo.asia/announcements/viblo-mobile-app-chinh-thuc-ra-mat-trai-nghiem-ngay-va-tham-gia-minigame-hap-dan-GyZJZo7GLjm',
  'tags': []},
 {'name': '[Homelab] #19 Cài đặt Apache Guacamole',
  'link': 'https://viblo.asia/p/homelab-19-cai-dat-apache-guacamole-pPLkNN3ZJRZ',
  'tags': ['homelab']},
 {'name': 'Phân tách và phối hợp Agent chuyên môn trong hệ thống AI hiện đại',
  'link': 'https://viblo.asia/p/phan-tach-va-phoi-hop-agent-chuyen-mon-trong-he-thong-ai-hien-dai-37Ldee8gVov',
  'tags': ['AI Agent',
   'aiagentchuyentrach',
   'phantachaiagent',
   'phantachvaphoihopaiagent']},
 {'name': 'Để tìm hiểu sơ đồ xương cá kỹ lưỡng, bạn chỉ cần đọc bài viết này nhé!',
  'link': 'https://viblo.asia/p/de-tim-hieu-so-do-xuong-ca-ky-luong-ban-chi-can-doc-bai-viet-nay-nhe-R5JRQQz04Gv',
  'tags': ['Biểu đồ xương cá', 'Bản đồ tư duy', 'ProcessOn']},
 {'name': 'Cách quản lý và truy x